# LA Studio Voice Cloning - VieNeu-TTS v3 Turbo

This notebook loads exactly `vieneu-tts-v3-turbo` (`pnnbao-ump/VieNeu-TTS-v3-Turbo`) on CUDA.
It is independent from API Gateway and refuses every other model ID.

1. Choose **Runtime -> Change runtime type -> GPU**.
2. Run all cells.
3. Copy the printed URL and token into LA Studio's Voice Cloning panel.


In [ ]:
!nvidia-smi
%pip install -q "torch==2.8.0" "torchaudio==2.8.0" --index-url https://download.pytorch.org/whl/cu128
%pip install -q --upgrade --force-reinstall --no-deps "torchvision==0.23.0" --index-url https://download.pytorch.org/whl/cu128
%pip install -q "transformers==4.57.6" "git+https://github.com/pnnbao97/VieNeu-TTS.git@f56ce97ffb37" "soundfile==0.13.1" "python-multipart==0.0.20" "fastapi==0.115.12" "uvicorn==0.34.3"

# Colab can retain an older torchvision after torch is upgraded. Transformers
# then masks the binary mismatch as a missing PreTrainedModel/Qwen3 class.
import importlib.metadata as package_metadata
import traceback

import torch
import torchvision

print("PyTorch stack:", torch.__version__, torchvision.__version__)
assert torch.cuda.is_available(), "CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU."
assert package_metadata.version("torchvision").split("+")[0] == "0.23.0", "VieNeu requires torchvision 0.23.0 with torch 2.8.0."
try:
    from transformers import PreTrainedModel
    from transformers.models.qwen3.modeling_qwen3 import Qwen3ForCausalLM
except Exception as error:
    traceback.print_exc()
    raise RuntimeError(
        "The Colab PyTorch/Transformers stack is not importable for VieNeu. "
        "Restart the runtime, rerun this install cell, then run all cells again."
    ) from error
print("Transformers imports verified for VieNeu:", PreTrainedModel.__name__, Qwen3ForCausalLM.__name__)


In [ ]:
from pathlib import Path

WORKER = Path('/content/la_studio_voice_clone_worker.py')
WORKER.write_text('import torch\n\nfrom vieneu import Vieneu\n\nMODEL_ID = "vieneu-tts-v3-turbo"\nMODEL_NAME = "VieNeu-TTS v3 Turbo"\nUPSTREAM_MODEL = "pnnbao-ump/VieNeu-TTS-v3-Turbo"\nMODEL = Vieneu(mode="v3turbo", device="cuda", backend="pytorch", backbone_repo=UPSTREAM_MODEL)\n\ndef prepare_exact_profile(profile):\n    return MODEL.encode_reference(profile["ref_audio"], denoise=True)\n\ndef clone_with_exact_model(profile, request):\n    speaker_emb, ref_codes = profile["state"]\n    voice = {"speaker_emb": speaker_emb, "codes": ref_codes}\n    audio = MODEL.infer(text=request.text, voice=voice, denoise=False)\n    return audio, 48000\n\nimport io\nimport os\nimport shutil\nimport tempfile\nimport threading\nimport uuid\nfrom pathlib import Path\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile\nfrom fastapi.responses import Response\nfrom pydantic import BaseModel, Field\n\nif not torch.cuda.is_available():\n    raise RuntimeError("CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.")\n\nTOKEN = os.environ["LA_STUDIO_COLAB_VOICE_CLONE_TOKEN"]\nDATA_DIR = Path("/content/la-studio-voice-clone-data") / MODEL_ID\nDATA_DIR.mkdir(parents=True, exist_ok=True)\nMAX_REFERENCE_BYTES = 256 * 1024 * 1024\nMAX_INPUT_CHARS = 4000\nMAX_OUTPUT_SECONDS = 300\nMODEL_LOCK = threading.Lock()\nSTATE_LOCK = threading.Lock()\nPROFILES = {}\nJOBS = {}\n\nclass GenerationRequest(BaseModel):\n    model: str = Field(min_length=1, max_length=120)\n    profile_id: str = Field(min_length=1, max_length=160)\n    text: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\n    language: str = Field(default="vi", max_length=40)\n    speed: float = Field(default=1.0, ge=0.1, le=2.0)\n    num_step: int = Field(default=32, ge=1, le=64)\n\ndef authorize(authorization: str | None) -> None:\n    if authorization != "Bearer " + TOKEN:\n        raise HTTPException(status_code=401, detail="invalid worker token")\n\ndef require_exact_model(model: str) -> None:\n    if model.strip().lower() != MODEL_ID:\n        raise HTTPException(\n            status_code=409,\n            detail=f"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{model}\'. Open the notebook for the selected model.",\n        )\n\ndef audio_array(value):\n    if isinstance(value, (list, tuple)):\n        if not value:\n            raise RuntimeError("the selected model returned no audio")\n        value = value[0]\n    if torch.is_tensor(value):\n        value = value.detach().float().cpu().numpy()\n    audio = np.asarray(value, dtype=np.float32).reshape(-1)\n    if audio.size == 0 or not np.isfinite(audio).all():\n        raise RuntimeError("the selected model returned invalid audio")\n    return audio\n\ndef write_wav(path: Path, value, sample_rate: int) -> None:\n    audio = audio_array(value)\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\n        raise RuntimeError("generated audio exceeds the five minute output limit")\n    peak = float(np.max(np.abs(audio)))\n    if peak > 1.2:\n        audio = audio / peak\n    sf.write(path, audio, int(sample_rate), format="WAV", subtype="PCM_16")\n\ndef public_job(job_id: str):\n    with STATE_LOCK:\n        job = JOBS.get(job_id)\n        if not job:\n            raise HTTPException(status_code=404, detail="voice job not found")\n        return {key: value for key, value in job.items() if key not in {"audio_path", "cancelled"}}\n\ndef fail_job(job_id: str, error: Exception) -> None:\n    with STATE_LOCK:\n        job = JOBS.get(job_id)\n        if job:\n            job.update({\n                "status": "failed",\n                "stage": "failed",\n                "error": {"message": f"{type(error).__name__}: {str(error)[:300]}"},\n            })\n\ndef build_profile(job_id: str, profile_id: str) -> None:\n    try:\n        with STATE_LOCK:\n            profile = PROFILES[profile_id]\n            JOBS[job_id].update({"status": "running", "stage": "prepare_profile", "percent": 10})\n        with MODEL_LOCK:\n            state = prepare_exact_profile(profile)\n        with STATE_LOCK:\n            profile["state"] = state\n            JOBS[job_id].update({\n                "status": "succeeded",\n                "stage": "complete",\n                "percent": 100,\n                "result": {"id": profile_id, "model": MODEL_ID},\n            })\n    except Exception as error:\n        fail_job(job_id, error)\n\ndef generate_audio(job_id: str, request: GenerationRequest) -> None:\n    try:\n        with STATE_LOCK:\n            profile = PROFILES.get(request.profile_id)\n            if not profile:\n                raise RuntimeError("voice profile no longer exists")\n            JOBS[job_id].update({"status": "running", "stage": "generate", "percent": 10})\n        with MODEL_LOCK:\n            audio, sample_rate = clone_with_exact_model(profile, request)\n        output_path = DATA_DIR / f"{job_id}.wav"\n        write_wav(output_path, audio, sample_rate)\n        with STATE_LOCK:\n            if JOBS[job_id].get("cancelled"):\n                JOBS[job_id].update({"status": "cancelled", "stage": "cancelled", "percent": 0})\n                output_path.unlink(missing_ok=True)\n            else:\n                JOBS[job_id].update({\n                    "status": "succeeded",\n                    "stage": "complete",\n                    "percent": 100,\n                    "audio_path": str(output_path),\n                    "result": {"model": MODEL_ID, "sample_rate": int(sample_rate)},\n                })\n    except Exception as error:\n        fail_job(job_id, error)\n\napp = FastAPI(title=f"LA Studio Voice Cloning - {MODEL_NAME}", docs_url=None, redoc_url=None, openapi_url=None)\n\n@app.get("/health")\n@app.get("/v1/health")\ndef health(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "status": "ready",\n        "ready": True,\n        "device": "cuda",\n        "gpu": torch.cuda.get_device_name(0),\n        "model": MODEL_ID,\n        "variant": "fixed",\n        "upstream_model": UPSTREAM_MODEL,\n        "cpu_fallback": False,\n    }\n\n@app.get("/v1/capabilities")\ndef capabilities(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "contract_version": 1,\n        "device": "cuda",\n        "capabilities": [{\n            "id": "voice-cloning",\n            "models": [{\n                "id": MODEL_ID,\n                "name": MODEL_NAME,\n                "variant": "fixed",\n                "upstream_model": UPSTREAM_MODEL,\n                "formats": ["wav"],\n                "reference_formats": ["wav", "mp3", "flac"],\n                "reference_duration_seconds": {"min": 3, "max": 30},\n                "requires_consent": True,\n                "device": "cuda",\n                "loaded": True,\n            }],\n        }],\n    }\n\n@app.post("/v2/jobs/profile", status_code=202)\nasync def create_profile(\n    model: str = Form(...),\n    name: str = Form(...),\n    consent_confirmed: bool = Form(...),\n    ref_text: str = Form(default=""),\n    language: str = Form(default="vi"),\n    separate_music: bool = Form(default=False),\n    ref_audio: UploadFile = File(...),\n    authorization: str | None = Header(default=None),\n):\n    authorize(authorization)\n    require_exact_model(model)\n    if not consent_confirmed:\n        raise HTTPException(status_code=403, detail="explicit voice-cloning consent is required")\n    if not name.strip():\n        raise HTTPException(status_code=422, detail="profile name is required")\n    suffix = Path(ref_audio.filename or "").suffix.lower()\n    if suffix not in {".wav", ".mp3", ".flac"}:\n        raise HTTPException(status_code=415, detail="reference audio must be WAV, MP3, or FLAC")\n    profile_id = uuid.uuid4().hex\n    reference_path = DATA_DIR / f"{profile_id}{suffix}"\n    size = 0\n    with reference_path.open("wb") as output:\n        while chunk := await ref_audio.read(1024 * 1024):\n            size += len(chunk)\n            if size > MAX_REFERENCE_BYTES:\n                reference_path.unlink(missing_ok=True)\n                raise HTTPException(status_code=413, detail="reference audio exceeds 256 MB")\n            output.write(chunk)\n    try:\n        info = sf.info(reference_path)\n        duration = float(info.frames) / float(info.samplerate)\n    except Exception as error:\n        reference_path.unlink(missing_ok=True)\n        raise HTTPException(status_code=422, detail=f"reference audio cannot be decoded: {error}") from error\n    if duration < 3.0 or duration > 30.0:\n        reference_path.unlink(missing_ok=True)\n        raise HTTPException(status_code=422, detail="reference audio must be between 3 and 30 seconds")\n    job_id = uuid.uuid4().hex\n    profile = {\n        "id": profile_id,\n        "model": MODEL_ID,\n        "name": name.strip(),\n        "ref_audio": str(reference_path),\n        "ref_text": ref_text.strip(),\n        "language": language.strip() or "vi",\n        "separate_music": bool(separate_music),\n    }\n    with STATE_LOCK:\n        PROFILES[profile_id] = profile\n        JOBS[job_id] = {"id": job_id, "status": "queued", "stage": "queued", "percent": 0}\n    threading.Thread(target=build_profile, args=(job_id, profile_id), daemon=True).start()\n    return public_job(job_id)\n\n@app.post("/v2/jobs/generation", status_code=202)\ndef create_generation(request: GenerationRequest, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    require_exact_model(request.model)\n    with STATE_LOCK:\n        profile = PROFILES.get(request.profile_id)\n        if not profile:\n            raise HTTPException(status_code=404, detail="voice profile not found")\n        if profile["model"] != MODEL_ID:\n            raise HTTPException(status_code=409, detail="voice profile belongs to a different model worker")\n        job_id = uuid.uuid4().hex\n        JOBS[job_id] = {"id": job_id, "status": "queued", "stage": "queued", "percent": 0}\n    threading.Thread(target=generate_audio, args=(job_id, request), daemon=True).start()\n    return public_job(job_id)\n\n@app.get("/v2/jobs/{job_id}/audio")\ndef job_audio(job_id: str, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    with STATE_LOCK:\n        job = JOBS.get(job_id)\n        path = Path(job.get("audio_path", "")) if job else None\n    if not job:\n        raise HTTPException(status_code=404, detail="voice job not found")\n    if job.get("status") != "succeeded" or not path or not path.is_file():\n        raise HTTPException(status_code=409, detail="voice job audio is not ready")\n    return Response(path.read_bytes(), media_type="audio/wav", headers={"Cache-Control": "no-store"})\n\n@app.get("/v2/jobs/{job_id}")\ndef job_status(job_id: str, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return public_job(job_id)\n\n@app.delete("/v2/jobs/{job_id}")\ndef cancel_job(job_id: str, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    with STATE_LOCK:\n        job = JOBS.get(job_id)\n        if not job:\n            raise HTTPException(status_code=404, detail="voice job not found")\n        job["cancelled"] = True\n        if job["status"] == "queued":\n            job.update({"status": "cancelled", "stage": "cancelled", "percent": 0})\n    return {"cancelled": True}\n\n@app.delete("/v1/profiles/{profile_id}")\ndef delete_profile(profile_id: str, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    with STATE_LOCK:\n        profile = PROFILES.pop(profile_id, None)\n    if profile:\n        Path(profile["ref_audio"]).unlink(missing_ok=True)\n    return {"deleted": bool(profile)}\n', encoding='utf-8')
print('Worker source:', WORKER)


In [ ]:
# LA Studio worker launch contract: launch-2026-08-06.1
import json
import os
import queue
import re
import secrets
import signal
import socket
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path

CAPABILITY_LABEL = 'Voice Cloning'
MODEL_ID = 'vieneu-tts-v3-turbo'
PORT = 3923
TOKEN_ENV = 'LA_STUDIO_COLAB_VOICE_CLONE_TOKEN'
URL_ENV = 'LA_STUDIO_COLAB_VOICE_CLONE_URL'
MODEL_ENV = 'LA_STUDIO_COLAB_VOICE_CLONE_MODEL'
WORKER_LOG = Path('/content/la_studio_voice_clone_worker.log')
WORKER_MODULE = 'la_studio_voice_clone_worker'
WORKER_PYTHON = sys.executable
WORKER_PYTHON_ISOLATED = False
WORKER_ENVIRONMENT = {}
REQUIRES_CUDA = True
STARTUP_TIMEOUT_SECONDS = 20 * 60
TUNNEL_TIMEOUT_SECONDS = 90
TOKEN = secrets.token_urlsafe(32)


def port_is_occupied(port: int) -> bool:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=0.5):
            return True
    except OSError:
        return False


def process_cmdline(pid: int) -> str:
    """Read a Linux process command line without depending on psutil."""
    try:
        return Path(f"/proc/{pid}/cmdline").read_bytes().replace(b"\0", b" ").decode(
            "utf-8", errors="replace"
        ).strip()
    except (FileNotFoundError, PermissionError, ProcessLookupError):
        return ""


def all_processes():
    for entry in Path("/proc").iterdir():
        if not entry.name.isdigit():
            continue
        pid = int(entry.name)
        command = process_cmdline(pid)
        if command:
            yield pid, command


def listening_processes(port: int) -> dict[int, str]:
    """Return PIDs listening on a local TCP port via /proc socket ownership."""
    target_port = f"{port:04X}"
    socket_inodes = set()
    for table_name in ("/proc/net/tcp", "/proc/net/tcp6"):
        try:
            lines = Path(table_name).read_text(encoding="utf-8").splitlines()[1:]
        except FileNotFoundError:
            continue
        for line in lines:
            fields = line.split()
            if len(fields) < 10:
                continue
            local_address, state, inode = fields[1], fields[3], fields[9]
            if state == "0A" and local_address.rsplit(":", 1)[-1].upper() == target_port:
                socket_inodes.add(inode)
    if not socket_inodes:
        return {}

    listeners = {}
    for entry in Path("/proc").iterdir():
        if not entry.name.isdigit():
            continue
        try:
            descriptors = (entry / "fd").iterdir()
        except (FileNotFoundError, PermissionError):
            continue
        for descriptor in descriptors:
            try:
                target = os.readlink(descriptor)
            except (FileNotFoundError, PermissionError, OSError):
                continue
            match = re.fullmatch(r"socket:\\[(\\d+)\\]", target)
            if match and match.group(1) in socket_inodes:
                pid = int(entry.name)
                listeners[pid] = process_cmdline(pid)
                break
    return listeners


def stop_pid(pid: int) -> None:
    if pid == os.getpid():
        return
    try:
        os.kill(pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    deadline = time.monotonic() + 10
    while time.monotonic() < deadline:
        try:
            os.kill(pid, 0)
        except ProcessLookupError:
            return
        time.sleep(0.2)
    try:
        os.kill(pid, signal.SIGKILL)
    except ProcessLookupError:
        pass


def reclaim_previous_la_studio_worker() -> None:
    """Stop only an older LA Studio worker/tunnel for this exact local port.

    Re-running a Colab cell keeps child processes alive.  The previous launch
    created a new token but aborted before it could replace the old worker,
    forcing users to destroy the whole GPU runtime.  We identify ownership by
    the exact generated module name and never terminate a foreign listener.
    """
    stopped = []
    for pid, command in listening_processes(PORT).items():
        if WORKER_MODULE in command and "uvicorn" in command:
            stop_pid(pid)
            stopped.append(f"worker PID {pid}")

    endpoint = f"http://127.0.0.1:{PORT}"
    for pid, command in all_processes():
        if ("cloudflared" in command and "tunnel" in command and endpoint in command):
            stop_pid(pid)
            stopped.append(f"tunnel PID {pid}")

    deadline = time.monotonic() + 12
    while port_is_occupied(PORT) and time.monotonic() < deadline:
        time.sleep(0.2)
    if stopped:
        print("Stopped previous LA Studio " + ", ".join(stopped) + ".")

    if port_is_occupied(PORT):
        listeners = listening_processes(PORT)
        foreign_pids = sorted(listeners) or ["unknown"]
        raise RuntimeError(
            f"Port {PORT} is occupied by a process that is not the previous LA Studio "
            f"{CAPABILITY_LABEL} worker (PID(s): {', '.join(map(str, foreign_pids))}). "
            "Choose a fresh Colab runtime rather than terminating an unrelated process."
        )


def worker_log_tail() -> str:
    try:
        return WORKER_LOG.read_text(encoding="utf-8", errors="replace")[-12000:]
    except FileNotFoundError:
        return "(worker log was not created)"


def stop_process(process) -> None:
    if process is None or process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()


reclaim_previous_la_studio_worker()

env = os.environ.copy()
env[TOKEN_ENV] = TOKEN
env["PYTHONUNBUFFERED"] = "1"
env.update(WORKER_ENVIRONMENT)
if WORKER_PYTHON_ISOLATED:
    # Do not let Colab's global site-packages or a notebook-level PYTHONPATH
    # bleed into a dedicated worker virtual environment.
    env.pop("PYTHONPATH", None)
    env["PYTHONNOUSERSITE"] = "1"
worker = None
tunnel = None

with WORKER_LOG.open("w", encoding="utf-8", buffering=1) as worker_output:
    worker = subprocess.Popen(
        [WORKER_PYTHON, "-m", "uvicorn", 'la_studio_voice_clone_worker:app', "--host", "127.0.0.1", "--port", str(PORT)],
        cwd="/content",
        env=env,
        stdout=worker_output,
        stderr=subprocess.STDOUT,
    )
    worker_kind = "exact CUDA" if REQUIRES_CUDA else "dedicated Colab CPU"
    print(f"Starting {worker_kind} {CAPABILITY_LABEL} worker.")
    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS
    last_error = "worker has not answered /health yet"
    next_report = time.monotonic()
    while time.monotonic() < deadline:
        exit_code = worker.poll()
        if exit_code is not None:
            raise RuntimeError(
                f"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\n\n"
                "---- LA Studio worker log (last 12,000 characters) ----\n" + worker_log_tail()
            )
        try:
            request = urllib.request.Request(
                f"http://127.0.0.1:{PORT}/health",
                headers={"Authorization": "Bearer " + TOKEN},
            )
            with urllib.request.urlopen(request, timeout=10) as response:
                health = json.loads(response.read().decode("utf-8"))
            if (response.status == 200
                    and health.get("ready") is True
                    and str(health.get("device", "")).lower()
                        == ("cuda" if REQUIRES_CUDA else "colab-cpu")
                    and str(health.get("model", "")).strip().lower() == MODEL_ID
                    and health.get("cpu_fallback") is False):
                print(worker_kind.title() + " worker is ready:", health)
                break
            last_error = "unexpected /health response: " + json.dumps(health, ensure_ascii=False)
        except urllib.error.HTTPError as error:
            last_error = f"/health returned HTTP {error.code}: " + error.read().decode("utf-8", errors="replace")[:1000]
        except Exception as error:
            last_error = f"/health is not ready: {type(error).__name__}: {error}"
        if time.monotonic() >= next_report:
            print(f"Waiting for the {worker_kind} worker...", last_error)
            next_report = time.monotonic() + 30
        time.sleep(2)
    else:
        stop_process(worker)
        raise RuntimeError(
            f"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within "
            f"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\n\n"
            "---- LA Studio worker log (last 12,000 characters) ----\n" + worker_log_tail()
        )


def cloudflared_ready() -> bool:
    try:
        return subprocess.run(
            ["cloudflared", "--version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            check=False,
        ).returncode == 0
    except OSError:
        return False


def ensure_cloudflared() -> None:
    if cloudflared_ready():
        return
    package_path = "/content/la-studio-cloudflared.deb"
    download = subprocess.run(
        [
            "curl", "--fail", "--location", "--retry", "4", "--retry-all-errors",
            "--output", package_path,
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    if download.returncode != 0:
        detail = download.stdout[-1200:].strip() or "no download output"
        raise RuntimeError("Could not download cloudflared: " + detail)
    install = subprocess.run(
        ["dpkg", "-i", package_path], text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, check=False,
    )
    if install.returncode != 0 or not cloudflared_ready():
        detail = install.stdout[-1200:].strip() or "no installation output"
        raise RuntimeError("Could not install cloudflared: " + detail)


ensure_cloudflared()
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
tunnel_lines = queue.Queue()


def collect_tunnel_output() -> None:
    assert tunnel.stdout is not None
    for line in tunnel.stdout:
        tunnel_lines.put(line)


threading.Thread(target=collect_tunnel_output, daemon=True).start()
public_url = ""
recent_tunnel_lines = []
deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS
while time.monotonic() < deadline and not public_url:
    if tunnel.poll() is not None:
        break
    try:
        line = tunnel_lines.get(timeout=1)
    except queue.Empty:
        continue
    recent_tunnel_lines.append(line.rstrip())
    recent_tunnel_lines = recent_tunnel_lines[-10:]
    print(line, end="")
    match = re.search(r"https://[^\s\"']+\.trycloudflare\.com", line)
    if match:
        # The desktop Check Colab action is the authoritative public endpoint,
        # bearer-token, capability, and exact-model verification.
        public_url = match.group(0)

if not public_url:
    stop_process(tunnel)
    stop_process(worker)
    tail = "\n".join(recent_tunnel_lines) or "(no cloudflared output)"
    raise RuntimeError(
        f"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\n"
        "---- cloudflared output ----\n" + tail
    )

os.environ[URL_ENV] = public_url
os.environ[TOKEN_ENV] = TOKEN
os.environ[MODEL_ENV] = MODEL_ID
print("\nLA Studio exact-model Colab worker is ready")
print(URL_ENV + "=" + public_url)
print(TOKEN_ENV + "=" + TOKEN)
print(MODEL_ENV + "=" + MODEL_ID)
print("Click Check Colab in the matching LA Studio feature before running it.")
